# Skin Disease Classification using Deep Learning (HAM10000)

This notebook trains a deep learning model to classify dermatoscopic images into 7 skin disease categories using the HAM10000 dataset.

**Project Objectives:**
- Download and explore the HAM10000 dataset.
- Perform group-aware data splitting to prevent data leakage (using `lesion_id`).
- Address class imbalance.
- Train a MobileNetV2 transfer-learning model.
- Evaluate the model comprehensively (F1, Confusion Matrix).
- Export the model for Flask web deployment.

> **Note on GPU:** Ensure you are running this notebook with a GPU accelerator (Runtime -> Change runtime type -> T4 GPU).

## 1. Install Dependencies and Setup Environment

In [ ]:
!pip install --upgrade pandas numpy matplotlib seaborn scikit-learn tensorflow pillow tf-keras

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 2. Dataset Download (Kaggle)

To download the dataset directly in Colab, you need a `kaggle.json` file. 
Upload it to Colab, or manually download the HAM10000 dataset and place the metadata file and images in the `dataset` folder.

In [ ]:
# Uncomment and run if you have uploaded kaggle.json
"""
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d dataset
"""

## 3. Metadata Exploration & Group-Aware Split

**CRITICAL:** The HAM10000 dataset contains multiple images for the same lesion (tracked by `lesion_id`). If we do a random split, images of the exact same lesion will end up in both the train and test sets, causing **data leakage** and artificially high accuracy. We must split by `lesion_id`.

In [ ]:
# Load Metadata
# Ensure you have 'HAM10000_metadata.csv' and the image folders extracted in './dataset/'
# Adjust paths as necessary based on your zip extraction

metadata_path = './dataset/HAM10000_metadata.csv'
if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print(f"Total images: {len(df)}")
    
    # Map class to a numeric label
    classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
    class_map = {cls: idx for idx, cls in enumerate(classes)}
    df['label'] = df['dx'].map(class_map)
    
    # Locate image paths
    # Images might be in 'HAM10000_images_part_1' and 'HAM10000_images_part_2'
    import glob
    image_paths = {os.path.splitext(os.path.basename(x))[0]: x for x in glob.glob('./dataset/**/*.jpg', recursive=True)}
    df['image_path'] = df['image_id'].map(image_paths.get)
    
    # Remove rows where image_path is None (missing images)
    df = df.dropna(subset=['image_path'])
    print(f"Total valid images: {len(df)}")
else:
    print("Dataset not found. Please ensure the dataset is extracted to ./dataset/")
    # Creating dummy dataframe for notebook validation if data is missing
    print("Creating dummy dataframe for demonstration purposes...")
    df = pd.DataFrame({'lesion_id': ['L1', 'L2'], 'image_id': ['img1', 'img2'], 'dx': ['nv', 'mel'], 'image_path': ['dummy1.jpg', 'dummy2.jpg'], 'label': [5, 4]})


In [ ]:
# Group-aware splitting by lesion_id
lesion_df = df.groupby('lesion_id')['dx'].first().reset_index()

# Split lesion_ids into train (80%), test(10%), val(10%)
train_lesions, test_val_lesions = train_test_split(lesion_df, test_size=0.2, stratify=lesion_df['dx'], random_state=SEED)
val_lesions, test_lesions = train_test_split(test_val_lesions, test_size=0.5, stratify=test_val_lesions['dx'], random_state=SEED)

# Map back to original dataframe to get image rows
train_df = df[df['lesion_id'].isin(train_lesions['lesion_id'])]
val_df = df[df['lesion_id'].isin(val_lesions['lesion_id'])]
test_df = df[df['lesion_id'].isin(test_lesions['lesion_id'])]

print(f"Train size: {len(train_df)} images")
print(f"Validation size: {len(val_df)} images")
print(f"Test size: {len(test_df)} images")

## 4. Handle Class Imbalance
HAM10000 is highly imbalanced (most images are Melanocytic Nevi - `nv`). We will use class weights during training.

In [ ]:
if len(train_df) > 10:
    class_weights = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    print("Class Weights:", class_weight_dict)
else:
    class_weight_dict = None
    print("Not enough data to compute class weights.")

## 5. Data Generators and Augmentation

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# MobileNetV2 expects inputs mapped between -1 and 1
def preprocess_func(img):
    return (img / 127.5) - 1.0

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_func,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_func
)

# In a real scenario, use flow_from_dataframe.
# Ensure df string columns are correctly setup.
# train_generator = train_datagen.flow_from_dataframe(
#     train_df, x_col='image_path', y_col='dx', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
# )


## 6. Model Architecture (MobileNetV2 Transfer Learning)

In [ ]:
def create_model():
    base_model = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_shape=(224, 224, 3)
    )
    
    # Freeze the base model for Stage 1
    base_model.trainable = False
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(7, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base_model

model, base_model = create_model()
model.summary()

## 7. Training

Due to time limits, we will do a brief training loop. In reality, you'd run this for 10-20 epochs, then unfreeze the base model for fine-tuning.

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_loss'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

# Uncomment to train when dataset is loaded
"""
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks
)
"""
print("Training logic defined.")

## 8. Fine-tuning (Stage 2)

In [ ]:
"""
base_model.trainable = True

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5), # Lower learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks
)
"""

## 9. Evaluation

In [ ]:
"""
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()
"""

## 10. Save and Export Model
The model `trained_model.keras` should be copied to your Flask application's `model/` folder.

In [ ]:
# If using Colab, you can save and download it:
"""
model.save('trained_model.keras')
from google.colab import files
files.download('trained_model.keras')
"""